In [5]:
import re
import csv

# مسیرها
input_path = r"F:\Thesis\project\2-RAG\raw_laws\civil\hemayat_honarmandan.txt"
output_tsv = r"F:\Thesis\project\2-RAG\raw_laws\civil\hemayat_honarmandan_articles.tsv"

LAW_CODE = "honar_moalef"
LAW_NAME = "قانون حمایت حقوق مولفان و مصنفان و هنرمندان"


def normalize_digits(text: str) -> str:
    persian_digits = "۰۱۲۳۴۵۶۷۸۹"
    english_digits = "0123456789"
    trans_table = str.maketrans(persian_digits, english_digits)
    return text.translate(trans_table)


def normalize_persian(text: str) -> str:
    # نرمال‌سازی حداقلی فارسی
    text = text.replace("ي", "ی").replace("ك", "ک")
    # فشرده‌سازی فاصله‌ها
    text = re.sub(r"\s+", " ", text)
    return text.strip()


# 1) خواندن و نرمال‌سازی اولیه
with open(input_path, "r", encoding="utf-8") as f:
    raw = f.read()

raw = normalize_digits(raw)
raw = raw.replace("–", "-").replace("—", "-")


# 2) اطمینان از اینکه هر «فصل ...» و هر «ماده N-» در ابتدای خط جدید باشد
#    (در صورتی که از HTML یا متن به‌هم‌چسبیده آمده باشد)
#   - قبل از هر "فصل " اگر در وسط خط است، \n بگذاریم
#   - قبل از هر "ماده N-" اگر در وسط خط است، \n بگذاریم
#   - توجه: از lookbehind استفاده می‌کنیم تا ابتدای واقعی خط را خراب نکنیم

# شکستن قبل از "فصل ..."
raw = re.sub(r"(?<!^)(فصل\s+[^ \n]+\s*-\s*)", r"\n\1", raw, flags=re.MULTILINE)

# شکستن قبل از "ماده N-"
raw = re.sub(r"(?<!^)(ماده\s+\d+\s*-\s*)", r"\n\1", raw, flags=re.MULTILINE)

lines = raw.splitlines()

# 3) الگوها: فصل و ماده
# مثال فصل‌ها:
#   "فصل  اول – کلیات"
#   "فصل دوم – میزان اجاره‌بها و ترتیب پرداخت آن"
chapter_pattern = re.compile(r"^\s*فصل\s+(.+)$")

# مثال ماده:
#   "ماده 1- هر محلی که ..."
article_pattern = re.compile(
    r"^\s*ماده\s*‌?\s*(\d+)\s*[–\-ـ]\s*(.*)$"
)
current_chapter = ""
articles = []

current_article_num = None
current_article_text = ""


for line in lines:
    stripped = line.strip()
    if not stripped:
        continue

    # 1) تشخیص فصل
    m_chap = chapter_pattern.match(stripped)
    if m_chap:
        # اگر مادهٔ باز داریم، اول آن را نهایی کنیم
        if current_article_num is not None and current_article_text:
            one_line = normalize_persian(current_article_text)
            articles.append(
                {
                    "law_code": LAW_CODE,
                    "law_name": LAW_NAME,
                    "chapter": current_chapter,
                    "article_number": current_article_num,
                    "text": one_line,
                }
            )
            current_article_num = None
            current_article_text = ""

        current_chapter = normalize_persian(stripped)
        continue

    # 2) تشخیص شروع ماده
    m_art = article_pattern.match(stripped)
    if m_art:
        # اگر یک مادهٔ قبلی باز است، نهایی کنیم
        if current_article_num is not None and current_article_text:
            one_line = normalize_persian(current_article_text)
            articles.append(
                {
                    "law_code": LAW_CODE,
                    "law_name": LAW_NAME,
                    "chapter": current_chapter,
                    "article_number": current_article_num,
                    "text": one_line,
                }
            )

        num_str = m_art.group(1)
        rest_text = m_art.group(2).strip()

        try:
            current_article_num = int(num_str)
        except ValueError:
            current_article_num = None

        # شروع متن ماده با خود «ماده N-»
        if rest_text:
            current_article_text = f"ماده {num_str}- {rest_text}"
        else:
            current_article_text = f"ماده {num_str}-"
        continue

    # 3) ادامهٔ متن مادهٔ جاری
    if current_article_num is not None:
        current_article_text += " " + stripped

# 4) بستن آخرین ماده
if current_article_num is not None and current_article_text:
    one_line = normalize_persian(current_article_text)
    articles.append(
        {
            "law_code": LAW_CODE,
            "law_name": LAW_NAME,
            "chapter": current_chapter,
            "article_number": current_article_num,
            "text": one_line,
        }
    )

# 5) نوشتن خروجی TSV
fieldnames = ["law_code", "law_name", "chapter", "article_number", "text"]

with open(output_tsv, "w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames, delimiter="\t")
    writer.writeheader()
    for row in articles:
        writer.writerow(row)

print("✓ پردازش mojer-mostajer-56 تمام شد.")
print(f"✓ تعداد مواد استخراج شده: {len(articles)}")
print(f"✓ خروجی TSV: {output_tsv}")


✓ پردازش mojer-mostajer-56 تمام شد.
✓ تعداد مواد استخراج شده: 32
✓ خروجی TSV: F:\Thesis\project\2-RAG\raw_laws\civil\hemayat_honarmandan_articles.tsv
